# Phase 5b: Brand Relevance

In [1]:
import pandas as pd
import numpy as np
import yaml
import json
import pickle
import os
import sys

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

sys.path.append(os.path.abspath('../src'))
from relevance import (
    get_brand_text, 
    compute_centroid_embeddings, 
    compute_hashtag_embeddings, 
    predict_category, 
    calculate_brand_relevance,
    get_hashtag_text
)


## 1. Configuration

In [2]:
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Hardcode the specific requirements for Phase 5b
config['centroid_sample'] = 500
config['hashtag_caption_sample'] = 20

# Save back to config
with open('../config.yaml', 'w') as f:
    yaml.dump(config, f)

print(f"Centroid Sample: {config['centroid_sample']}")
print(f"Hashtag Caption Sample: {config['hashtag_caption_sample']}")
print(f"Seed: {config['seed']}")


Centroid Sample: 500
Hashtag Caption Sample: 20
Seed: 42


## 2. Data Loading

In [3]:
posts = pd.read_parquet('../data/processed/posts_clean.parquet')
hashtags = pd.read_parquet('../data/processed/hashtags_clean.parquet')
features = pd.read_parquet('../data/processed/features_cold.parquet')[['post_id', 'split', 'original_category']]

posts = posts.merge(features, on='post_id', how='inner')
print(f"Loaded {len(posts)} posts.")

trends = pd.read_parquet('../artifacts/v1.0/trends.parquet')
eligible_hashtags = trends['hashtag'].unique()
print(f"Eligible hashtags for embedding: {len(eligible_hashtags)}")


Loaded 1313584 posts.
Eligible hashtags for embedding: 3405


## 3. Category Centroids

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

posts_train = posts[posts['split'] == 'train']
print(f"Training posts for centroids: {len(posts_train)}")

eval_centroids = compute_centroid_embeddings(posts_train, model, config['centroid_sample'], config['seed'])
shipped_centroids = compute_centroid_embeddings(posts, model, config['centroid_sample'], config['seed'])

with open('../artifacts/v1.0/category_centroids.pkl', 'wb') as f:
    pickle.dump(shipped_centroids, f)
    
print(f"Computed centroids for {len(eval_centroids)} categories.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Training posts for centroids: 1050869


Computed centroids for 9 categories.


## 4. Hashtag Embedding Construction

In [5]:
# We use the optimized group_by approach in compute_hashtag_embeddings
# Hashtag text = hashtag + sampled captions
ht_embs_df = compute_hashtag_embeddings(eligible_hashtags, hashtags, posts, model, config['hashtag_caption_sample'], config['seed'])

ht_embs_df.to_parquet('../artifacts/v1.0/hashtag_embeddings.parquet', index=False)
print(f"Saved embeddings for {len(ht_embs_df)} hashtags.")


Saved embeddings for 3405 hashtags.


## 5. TF-IDF Baseline Preparation

In [6]:
print("Preparing TF-IDF Baseline representations...")
tf_texts = []
tf_ht_list = []

valid_hashtags = hashtags[hashtags['hashtag'].isin(set(eligible_hashtags))]
grouped = valid_hashtags.groupby('hashtag')
post_captions = posts.set_index('post_id')['caption']

for h in eligible_hashtags:
    try:
        ph = grouped.get_group(h)
    except KeyError:
        ph = pd.DataFrame(columns=hashtags.columns)
    
    n_sample = min(len(ph), config['hashtag_caption_sample'])
    if n_sample > 0:
        sampled = ph.sample(n=n_sample, random_state=config['seed'])
        caps = post_captions.reindex(sampled['post_id']).fillna('').tolist()
    else:
        caps = []
        
    text = get_hashtag_text(h, caps)
    tf_texts.append(text)
    tf_ht_list.append(h)

tf_ht = TfidfVectorizer(max_features=10000)
tf_ht.fit(tf_texts)
ht_tfidf_mat = tf_ht.transform(tf_texts)
print(f"TF-IDF Matrix shape: {ht_tfidf_mat.shape}")


Preparing TF-IDF Baseline representations...


TF-IDF Matrix shape: (3405, 10000)


## 6. Held-Out Evaluation (Category Mapping)

In [7]:
# Evaluate on validation accounts
posts_val = posts[posts['split'] == 'val']
val_users = posts_val[['username', 'category']].drop_duplicates()
val_users = val_users[val_users['category'] != 'other']

print(f"Validation users: {len(val_users)}")

# Build profile text for each validation user
val_profiles = []
user_caps = posts_val.groupby('username')['caption'].apply(lambda x: " ".join([str(c) for c in x.dropna().head(20)])).reset_index()
val_users = val_users.merge(user_caps, on='username')

minilm_correct = 0
minilm_scores = []

for _, row in val_users.iterrows():
    true_cat = row['category']
    bt = get_brand_text(row['caption'], '', '') # Treat caption as brand description
    
    # MiniLM prediction
    pred_cat, score, _ = predict_category(bt, model, eval_centroids, tau=0.0)
    if pred_cat == true_cat:
        minilm_correct += 1
        minilm_scores.append(score)
    
print(f"Evaluating MiniLM Category-Mapping Accuracy on held-out profiles...")
acc_minilm = minilm_correct / len(val_users)
print(f"MiniLM Category-Mapping Accuracy: {acc_minilm:.4f}")


Validation users: 2726


Evaluating MiniLM Category-Mapping Accuracy on held-out profiles...
MiniLM Category-Mapping Accuracy: 0.4387


In [8]:
# Prepare TF-IDF centroids properly for accuracy calculation
train_users = posts_train[posts_train['category'] != 'other']
tf_centroids = {}
for cat in eval_centroids.keys():
    if cat == 'other': continue
    cat_posts = train_users[train_users['category'] == cat]
    caps = cat_posts.sample(n=min(len(cat_posts), config['centroid_sample']), random_state=config['seed'])['caption'].dropna().tolist()
    text = " ".join(caps)
    tf_centroids[cat] = tf_ht.transform([text])

tf_correct = 0
for _, row in val_users.iterrows():
    true_cat = row['category']
    bt = get_brand_text(row['caption'], '', '')
    bt_vec = tf_ht.transform([bt])
    
    best_cat = None
    best_score = -1.0
    for cat, cvec in tf_centroids.items():
        sim = cosine_similarity(bt_vec, cvec)[0][0]
        if sim > best_score:
            best_score = sim
            best_cat = cat
    
    if best_cat == true_cat:
        tf_correct += 1

acc_tfidf = tf_correct / len(val_users)
print(f"TF-IDF Category-Mapping Accuracy: {acc_tfidf:.4f}")


TF-IDF Category-Mapping Accuracy: 0.4831


## 7. Tau Calculation

In [9]:
# Tau = 5th percentile of the best cosine among correctly mapped held-out account profiles
tau = np.percentile(minilm_scores, 5)
print(f"Calculated tau: {tau:.4f}")

config['tau'] = float(tau)
with open('../config.yaml', 'w') as f:
    yaml.dump(config, f)


Calculated tau: 0.3431


## 8. Test Brands Evaluation

In [10]:
test_brands = [
    {'id': 'B1', 'desc': 'Clean and organic skincare formulations', 'ind': 'skincare cosmetics', 'aud': 'women seeking natural beauty', 'exp': 'beauty'},
    {'id': 'B2', 'desc': 'Matching outfits and gear for parents and toddlers', 'ind': 'family apparel', 'aud': 'young parents, moms, dads', 'exp': 'family'},
    {'id': 'B3', 'desc': 'Trendy streetwear and oversized hoodies', 'ind': 'fashion apparel', 'aud': 'teens and young adults', 'exp': 'fashion'},
    {'id': 'B4', 'desc': 'High-protein meal replacement shakes', 'ind': 'fitness supplements', 'aud': 'gym goers, athletes', 'exp': 'fitness'},
    {'id': 'B5', 'desc': 'Gourmet meal kit delivery service', 'ind': 'food delivery', 'aud': 'busy professionals, home cooks', 'exp': 'food'},
    {'id': 'B6', 'desc': 'Minimalist scandinavian living room furniture', 'ind': 'home furniture', 'aud': 'homeowners, decorators', 'exp': 'interior'},
    {'id': 'B7', 'desc': 'Grain-free premium dog kibble', 'ind': 'pet food', 'aud': 'dog owners', 'exp': 'pet'},
    {'id': 'B8', 'desc': 'Guided backpacking tours in the alps', 'ind': 'travel tourism', 'aud': 'adventure seekers, hikers', 'exp': 'travel'},
    {'id': 'B9', 'desc': 'Enterprise cloud accounting software', 'ind': 'B2B SaaS', 'aud': 'CFOs, accountants', 'exp': 'off-topic: software'},
    {'id': 'B10', 'desc': 'Network penetration testing services', 'ind': 'cybersecurity', 'aud': 'IT administrators', 'exp': 'off-topic: cybersecurity'}
]

def is_relevant(brand_cat, hashtag):
    h = hashtag.lower()
    if brand_cat == 'beauty' and any(k in h for k in ['skin', 'beauty', 'makeup', 'glow', 'face', 'cosmetic']): return 1
    if brand_cat == 'family' and any(k in h for k in ['fam', 'mom', 'dad', 'parent', 'kid', 'child', 'baby']): return 1
    if brand_cat == 'fashion' and any(k in h for k in ['fashion', 'style', 'outfit', 'wear', 'dress', 'clothes', 'ootd']): return 1
    if brand_cat == 'fitness' and any(k in h for k in ['fit', 'gym', 'workout', 'train', 'muscle', 'health']): return 1
    if brand_cat == 'food' and any(k in h for k in ['food', 'eat', 'cook', 'recipe', 'meal', 'yum', 'delicious']): return 1
    if brand_cat == 'interior' and any(k in h for k in ['interior', 'home', 'decor', 'room', 'house', 'furniture']): return 1
    if brand_cat == 'pet' and any(k in h for k in ['dog', 'cat', 'pet', 'puppy', 'animal']): return 1
    if brand_cat == 'travel' and any(k in h for k in ['travel', 'trip', 'explore', 'wander', 'vacation']): return 1
    return 0

results = []
ph_all = hashtags.merge(posts[['post_id', 'username']], on='post_id')
users = posts[['username', 'category']].drop_duplicates()
categories = ['beauty', 'family', 'fashion', 'fitness', 'food', 'interior', 'pet', 'travel']

for b in test_brands:
    bt = get_brand_text(b['desc'], b['ind'], b['aud'])
    cat, score, match_type = predict_category(bt, model, eval_centroids, tau)
    
    # Filter hashtag pool based on mapping
    if cat == 'all' or cat not in categories:
        pool_ht = list(eligible_hashtags)
    else:
        pool_users = set(users[users['category'] == cat]['username'])
        pool_ht = list(set(ph_all[ph_all['username'].isin(pool_users)]['hashtag']))
    
    b_df = ht_embs_df[ht_embs_df['hashtag'].isin(pool_ht)].copy()
    b_relev = calculate_brand_relevance(bt, model, b_df)
    top_10 = b_relev.sort_values('relevance', ascending=False).head(10)['hashtag'].tolist()
    
    p10 = sum(is_relevant(b['exp'], h) for h in top_10) / 10.0 if not b['exp'].startswith('off-topic') else None
    
    results.append({
        'brand': b['id'],
        'expected': b['exp'],
        'mapped_cat': cat,
        'cat_score': score,
        'match_type': match_type,
        'p10': p10
    })

res_df = pd.DataFrame(results)
print('Synthetic Brand Evaluation (MiniLM):')
display(res_df)

valid_p10 = [r['p10'] for r in results if r['p10'] is not None]
print(f'Mean P@10 (MiniLM): {np.mean(valid_p10):.4f}')


Synthetic Brand Evaluation (MiniLM):


,brand,expected,mapped_cat,cat_score,match_type,p10
0,B1,beauty,beauty,0.398977,matched,0.7
1,B2,family,all,0.322544,fallback_all,0.3
2,B3,fashion,beauty,0.372223,matched,0.5
3,B4,fitness,all,0.263809,fallback_all,0.3
4,B5,food,food,0.380017,matched,0.3
5,B6,interior,interior,0.500451,matched,0.6
6,B7,pet,all,0.331089,fallback_all,0.3
7,B8,travel,all,0.230997,fallback_all,0.2
8,B9,off-topic: software,all,0.024795,fallback_all,NaN
9,B10,off-topic: cybersecurity,all,0.023536,fallback_all,NaN


Mean P@10 (MiniLM): 0.4000


In [11]:
# TF-IDF test brand evaluation
tf_results = []
for b in test_brands:
    if b['exp'].startswith('off-topic'): continue
    bt = get_brand_text(b['desc'], b['ind'], b['aud'])
    bt_vec = tf_ht.transform([bt])
    sims = cosine_similarity(bt_vec, ht_tfidf_mat)[0]
    top_10_idx = np.argsort(sims)[::-1][:10]
    top_10 = [tf_ht_list[i] for i in top_10_idx]
    p10 = sum(is_relevant(b['exp'], h) for h in top_10) / 10.0
    tf_results.append(p10)

print(f'Mean P@10 (TF-IDF): {np.mean(tf_results):.4f}')


Mean P@10 (TF-IDF): 0.5000


## 9. Validation of Outputs

In [12]:
# Verify hashtag embeddings schema
ht_embs_df = pd.read_parquet('../artifacts/v1.0/hashtag_embeddings.parquet')
print(ht_embs_df.info())
print(f"Dimensionality: {len(ht_embs_df.iloc[0]['embedding'])}")

with open('../artifacts/v1.0/category_centroids.pkl', 'rb') as f:
    shipped_centroids = pickle.load(f)
print(f"Shipped centroids for {len(shipped_centroids)} categories.")


<class 'pandas.DataFrame'>
RangeIndex: 3405 entries, 0 to 3404
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   hashtag    3405 non-null   str   
 1   embedding  3405 non-null   object
dtypes: object(1), str(1)
memory usage: 89.0+ KB
None
Dimensionality: 384
Shipped centroids for 9 categories.
